In [2]:

from google.colab import files

uploaded = files.upload()


Saving dataset.csv to dataset.csv


In [3]:

import pandas as pd

# Get the uploaded CSV filename
filename = next(iter(uploaded.keys()))

# Load the dataset
df = pd.read_csv(filename)

# Display dataset information
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nAvailable language labels:")
print(df.iloc[:, -1].value_counts())


Dataset shape: (22000, 2)

Column names:
['Text', 'language']

First 5 rows:


,Text,language
0,klement gottwaldi surnukeha palsameeriti ning ...,Estonian
1,sebes joseph pereira thomas på eng the jesuit...,Swedish
2,ถนนเจริญกรุง อักษรโรมัน thanon charoen krung เ...,Thai
3,விசாகப்பட்டினம் தமிழ்ச்சங்கத்தை இந்துப் பத்திர...,Tamil
4,de spons behoort tot het geslacht haliclona en...,Dutch



Available language labels:
language
Estonian      1000
Swedish       1000
Thai          1000
Tamil         1000
Dutch         1000
Japanese      1000
Turkish       1000
Latin         1000
Urdu          1000
Indonesian    1000
Portugese     1000
French        1000
Chinese       1000
Korean        1000
Hindi         1000
Spanish       1000
Pushto        1000
Persian       1000
Romanian      1000
Russian       1000
English       1000
Arabic        1000
Name: count, dtype: int64


In [4]:

# Keep only Tamil and English samples
df_two = df[df["language"].isin(["Tamil", "English"])].copy()

# Check the selected dataset
print("Dataset shape:", df_two.shape)

print("\nLanguage distribution:")
print(df_two["language"].value_counts())

print("\nFirst 5 samples:")
display(df_two.head())


Dataset shape: (2000, 2)

Language distribution:
language
Tamil      1000
English    1000
Name: count, dtype: int64

First 5 samples:


,Text,language
3,விசாகப்பட்டினம் தமிழ்ச்சங்கத்தை இந்துப் பத்திர...,Tamil
20,ஆங்கில எழுத்துக்களை மட்டும் அல்லாமல் பன்மொழி எ...,Tamil
37,in johnson was awarded an american institute ...,English
40,bussy-saint-georges has built its identity on ...,English
62,இரண்டு மீட்டர்களும் ஒரு அரை மீட்டரும் இணைந்த ந...,Tamil


In [5]:

import re

# Handle missing values
df_two = df_two.dropna(subset=["Text", "language"]).copy()

# Clean text while preserving Tamil Unicode characters
def clean_text(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text

df_two["Text"] = df_two["Text"].apply(clean_text)

# Remove empty text samples
df_two = df_two[df_two["Text"] != ""]

print("Dataset size after preprocessing:", len(df_two))
print("\nLanguage distribution:")
print(df_two["language"].value_counts())

display(df_two.head())


Dataset size after preprocessing: 2000

Language distribution:
language
Tamil      1000
English    1000
Name: count, dtype: int64


,Text,language
3,விசாகப்பட்டினம் தமிழ்ச்சங்கத்தை இந்துப் பத்திர...,Tamil
20,ஆங்கில எழுத்துக்களை மட்டும் அல்லாமல் பன்மொழி எ...,Tamil
37,in johnson was awarded an american institute o...,English
40,bussy-saint-georges has built its identity on ...,English
62,இரண்டு மீட்டர்களும் ஒரு அரை மீட்டரும் இணைந்த ந...,Tamil


In [6]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Separate text and language labels
X = df_two["Text"].values
y = df_two["language"].values

# Convert language labels to numbers
encoder = LabelEncoder()
y = encoder.fit_transform(y)

print("Language mapping:")
for i, language in enumerate(encoder.classes_):
    print(language, "=", i)

# Split into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Tokenize the text
tokenizer = Tokenizer(num_words=20000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Pad sequences to equal length
MAX_LEN = 100

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

print("\nTraining samples:", len(X_train_pad))
print("Testing samples:", len(X_test_pad))
print("Training input shape:", X_train_pad.shape)
print("Testing input shape:", X_test_pad.shape)


Language mapping:
English = 0
Tamil = 1

Training samples: 1600
Testing samples: 400
Training input shape: (1600, 100)
Testing input shape: (400, 100)


In [7]:

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

# Calculate vocabulary size
vocab_size = min(20000, len(tokenizer.word_index) + 1)

# Build the LSTM model
model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=64,
        mask_zero=True
    ),
    LSTM(64),
    Dropout(0.3),
    Dense(32, activation="relu"),
    Dense(2, activation="softmax")
])

# Compile the model
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Display model architecture
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [8]:

# Explicitly build the model for sequences of length 100
model.build(input_shape=(None, MAX_LEN))

# Display the model architecture
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 100, 64)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,315,170 (5.02 MB)

 Trainable params: 1,315,170 (5.02 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:

from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    X_train_pad,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)


Epoch 1/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 10s 164ms/step - accuracy: 0.7664 - loss: 0.4559 - val_accuracy: 0.9937 - val_loss: 0.1259
Epoch 2/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 4s 105ms/step - accuracy: 0.9922 - loss: 0.0657 - val_accuracy: 0.9750 - val_loss: 0.1470
Epoch 3/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 2s 60ms/step - accuracy: 0.9937 - loss: 0.0415 - val_accuracy: 0.9781 - val_loss: 0.1069
Epoch 4/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 3s 70ms/step - accuracy: 0.9953 - loss: 0.0302 - val_accuracy: 0.9781 - val_loss: 0.1143
Epoch 5/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 3s 81ms/step - accuracy: 0.9977 - loss: 0.0172 - val_accuracy: 0.9812 - val_loss: 0.1039
Epoch 6/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 4s 61ms/step - accuracy: 0.9977 - loss: 0.0159 - val_accuracy: 0.9812 - val_loss: 0.1054
Epoch 7/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 2s 61ms/step - accuracy: 0.9977 - loss: 0.0134 - val_accuracy: 0.9844 - val_loss: 0.0892
Epoch 8/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 3s 69ms/step - accuracy: 0.9992 - loss: 0.0053 - val_accuracy: 0.9844 

In [10]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import numpy as np

# Evaluate on test data
test_loss, test_accuracy = model.evaluate(
    X_test_pad,
    y_test,
    verbose=0
)

# Predict test samples
probabilities = model.predict(X_test_pad, verbose=0)
y_pred = np.argmax(probabilities, axis=1)

# Calculate evaluation metrics
print("TAMIL AND ENGLISH LANGUAGE IDENTIFICATION")
print("-----------------------------------------")
print(f"Test Loss : {test_loss:.4f}")
print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision : {precision_score(y_test, y_pred, average='macro', zero_division=0):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred, average='macro', zero_division=0):.4f}")
print(f"F1-Score  : {f1_score(y_test, y_pred, average='macro', zero_division=0):.4f}")

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1],
    target_names=encoder.classes_,
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


TAMIL AND ENGLISH LANGUAGE IDENTIFICATION
-----------------------------------------
Test Loss : 0.1544
Accuracy  : 0.9725
Precision : 0.9739
Recall    : 0.9725
F1-Score  : 0.9725

Classification Report:
              precision    recall  f1-score   support

     English       1.00      0.94      0.97       200
       Tamil       0.95      1.00      0.97       200

    accuracy                           0.97       400
   macro avg       0.97      0.97      0.97       400
weighted avg       0.97      0.97      0.97       400

Confusion Matrix:
[[189  11]
 [  0 200]]


In [11]:

def predict_language(text):
    sequence = tokenizer.texts_to_sequences([text])

    padded = pad_sequences(
        sequence,
        maxlen=MAX_LEN,
        padding="post",
        truncating="post"
    )

    probabilities = model.predict(padded, verbose=0)[0]
    predicted_index = np.argmax(probabilities)

    print("Input text:", text)
    print("Predicted language:", encoder.classes_[predicted_index])
    print(
        f"Prediction confidence: "
        f"{probabilities[predicted_index] * 100:.2f}%"
    )
    print("-" * 40)

# Test English
predict_language("Hello, how are you?")

# Test Tamil
predict_language("வணக்கம், நீங்கள் எப்படி இருக்கிறீர்கள்?")

# Another English example
predict_language("Machine learning helps computers learn from data.")

# Another Tamil example
predict_language("தமிழ் எங்கள் தாய்மொழி.")


Input text: Hello, how are you?
Predicted language: Tamil
Prediction confidence: 59.06%
----------------------------------------
Input text: வணக்கம், நீங்கள் எப்படி இருக்கிறீர்கள்?
Predicted language: Tamil
Prediction confidence: 75.37%
----------------------------------------
Input text: Machine learning helps computers learn from data.
Predicted language: Tamil
Prediction confidence: 55.67%
----------------------------------------
Input text: தமிழ் எங்கள் தாய்மொழி.
Predicted language: Tamil
Prediction confidence: 70.79%
----------------------------------------


In [12]:

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np

# 1. Character-level tokenization
char_tokenizer = Tokenizer(
    char_level=True,
    filters="",
    lower=False,
    oov_token="<OOV>"
)

char_tokenizer.fit_on_texts(X_train)

# 2. Convert characters into numerical sequences
X_train_char = char_tokenizer.texts_to_sequences(X_train)
X_test_char = char_tokenizer.texts_to_sequences(X_test)

# 3. Pad sequences
MAX_CHAR_LEN = 300

X_train_char_pad = pad_sequences(
    X_train_char,
    maxlen=MAX_CHAR_LEN,
    padding="post",
    truncating="post"
)

X_test_char_pad = pad_sequences(
    X_test_char,
    maxlen=MAX_CHAR_LEN,
    padding="post",
    truncating="post"
)

char_vocab_size = len(char_tokenizer.word_index) + 1

print("Character vocabulary size:", char_vocab_size)
print("Training shape:", X_train_char_pad.shape)
print("Testing shape:", X_test_char_pad.shape)

# 4. Build a new character-level LSTM
char_model = Sequential([
    Embedding(
        input_dim=char_vocab_size,
        output_dim=32,
        mask_zero=True
    ),
    LSTM(64),
    Dropout(0.3),
    Dense(32, activation="relu"),
    Dense(2, activation="softmax")
])

char_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

char_model.summary()

# 5. Train the new model
char_early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

char_history = char_model.fit(
    X_train_char_pad,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    callbacks=[char_early_stop],
    verbose=1
)


Character vocabulary size: 299
Training shape: (1600, 300)
Testing shape: (400, 300)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 12s 181ms/step - accuracy: 0.9750 - loss: 0.2750 - val_accuracy: 0.9969 - val_loss: 0.0187
Epoch 2/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 6s 153ms/step - accuracy: 0.9914 - loss: 0.0429 - val_accuracy: 0.9969 - val_loss: 0.0192
Epoch 3/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 6s 153ms/step - accuracy: 0.9914 - loss: 0.0423 - val_accuracy: 1.0000 - val_loss: 0.0068
Epoch 4/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 6s 143ms/step - accuracy: 0.9953 - loss: 0.0278 - val_accuracy: 1.0000 - val_loss: 0.0040
Epoch 5/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 7s 164ms/step - accuracy: 0.9953 - loss: 0.0268 - val_accuracy: 1.0000 - val_loss: 0.0037
Epoch 6/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 10s 168ms/step - accuracy: 0.9953 - loss: 0.0266 - val_accuracy: 1.0000 - val_loss: 0.0039
Epoch 7/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 5s 133ms/step - accuracy: 0.9953 - loss: 0.0248 - val_accuracy: 1.0000 - val_loss: 0.0024
Epoch 8/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 10s 133ms/step - accuracy: 0.9953 - loss: 0.0232 - val_accuracy:

In [13]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import numpy as np

# Predict the unseen test samples
char_probabilities = char_model.predict(
    X_test_char_pad,
    verbose=0
)

char_y_pred = np.argmax(char_probabilities, axis=1)

print("CHARACTER-LEVEL LSTM RESULTS")
print("----------------------------")
print(f"Accuracy  : {accuracy_score(y_test, char_y_pred):.4f}")
print(f"Precision : {precision_score(y_test, char_y_pred, average='macro', zero_division=0):.4f}")
print(f"Recall    : {recall_score(y_test, char_y_pred, average='macro', zero_division=0):.4f}")
print(f"F1-score  : {f1_score(y_test, char_y_pred, average='macro', zero_division=0):.4f}")

print("\nClassification Report:")
print(classification_report(
    y_test,
    char_y_pred,
    labels=[0, 1],
    target_names=encoder.classes_,
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, char_y_pred))


CHARACTER-LEVEL LSTM RESULTS
----------------------------
Accuracy  : 0.9925
Precision : 0.9926
Recall    : 0.9925
F1-score  : 0.9925

Classification Report:
              precision    recall  f1-score   support

     English       0.99      1.00      0.99       200
       Tamil       1.00      0.98      0.99       200

    accuracy                           0.99       400
   macro avg       0.99      0.99      0.99       400
weighted avg       0.99      0.99      0.99       400

Confusion Matrix:
[[200   0]
 [  3 197]]


In [14]:

def predict_language_char(text):
    sequence = char_tokenizer.texts_to_sequences([text])

    padded = pad_sequences(
        sequence,
        maxlen=MAX_CHAR_LEN,
        padding="post",
        truncating="post"
    )

    probs = char_model.predict(padded, verbose=0)[0]
    index = np.argmax(probs)

    print("Input text:", text)
    print("Predicted language:", encoder.classes_[index])
    print(f"Confidence: {probs[index] * 100:.2f}%")
    print("-" * 40)

predict_language_char("Hello, how are you?")
predict_language_char("வணக்கம், நீங்கள் எப்படி இருக்கிறீர்கள்?")
predict_language_char(
    "Machine learning helps computers learn from data."
)
predict_language_char("தமிழ் எங்கள் தாய்மொழி.")


Input text: Hello, how are you?
Predicted language: English
Confidence: 98.91%
----------------------------------------
Input text: வணக்கம், நீங்கள் எப்படி இருக்கிறீர்கள்?
Predicted language: Tamil
Confidence: 100.00%
----------------------------------------
Input text: Machine learning helps computers learn from data.
Predicted language: English
Confidence: 99.53%
----------------------------------------
Input text: தமிழ் எங்கள் தாய்மொழி.
Predicted language: Tamil
Confidence: 100.00%
----------------------------------------


In [15]:
import pickle

# Save the trained LSTM model
char_model.save("tamil_english_lstm.keras")

# Save the character tokenizer
with open("tamil_english_tokenizer.pkl", "wb") as f:
    pickle.dump(char_tokenizer, f)

# Save the label encoder
with open("tamil_english_encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)

print("All model files saved successfully!")

All model files saved successfully!


In [16]:
from google.colab import files

files.download("tamil_english_lstm.keras")
files.download("tamil_english_tokenizer.pkl")
files.download("tamil_english_encoder.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>